# H2 follow-up: score v3-r16 (v4) and v4-mixed-r16 (v5) on a SECOND held-out real-audio set

SESSIONS.md H2 -- `first10.wav` (career webinar) and tier4a (`real-meetings-bench`,
ML/VAE) gave OPPOSITE verdicts on whether v5 regresses on real audio. This
notebook adds a THIRD, independent real-audio point: 20 hand-corrected
segments of `NZiW4QH83CI` (a programming/RPA tech talk), from the
29:00-35:49 window of the video -- chosen for the highest code-switch
density of any 10-minute window (5.24% vs 4.15% for a naive first-10-min
cut; see `youtube-data-pilot/caption-probe.md`).

`NZiW4QH83CI` is not one of the 7 videos `mixed-noisy-v1` was built from
(`dataset/youtube-meetings/raw/` as of 2026-08-17) -- held out from both
models' training, same as `first10.wav`.

Reference: Google auto-caption (`vi-orig`), hand-corrected by the user for
misheard words + English spelling -- same quality tier as the `youtube-meetings`
training labels themselves (`scripts/review_youtube.py`), NOT an
independent professional transcript like `first10.wav`'s. 6 of the
originally-corrected 26 segments contain `<??>` (user could not make out
the audio) and are excluded -- 20 usable segments remain.

**GPU required (T4 is enough -- inference only, no training).**

**Before running:**
- Commit and push `scripts/eval_v3_v5_on_corrected_segments.py` to `main` first --
  Cell 1 clones `main`, not your local tree.
- Attach as Kaggle Dataset inputs (Add Data):
  - the existing **`h6-artifacts`** dataset (has `v3-r16/checkpoints/best` + `v3-r16/config.json`)
  - a **new dataset** zipped from this repo's `Outputs/h2-nziw-artifacts.zip`
    (v4-mixed-r16/checkpoints/best, v4-mixed-r16/config.json, the
    `NZiW4QH83CI` manifest + hand corrections + segment audio -- 123 MB)

**Disk:** ~3.1 GB base model download into `~/.cache/huggingface`. Nothing else
is written except two predictions CSVs.

## 1. Clone / update the repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the paths below -- mount paths
nest one level deeper than the attached dataset name, e.g.
`/kaggle/input/datasets/<user>/<slug>/<slug>`.

In [ ]:
!ls -la /kaggle/input
!ls -la /kaggle/input/datasets/*/*

## 3. Set platform-specific paths

Edit these to match what the cell above printed. This is the only place a
`/kaggle/input/...` path is written -- `scripts/eval_v3_v5_on_corrected_segments.py`
never hardcodes one.

In [ ]:
H6_ARTIFACTS = "/kaggle/input/datasets/<user>/h6-artifacts/h6-artifacts"                  # edit
H2_NZIW = "/kaggle/input/datasets/<user>/h2-nziw-artifacts/h2-nziw-artifacts"                # edit

V3_RUN_DIR = f"{H6_ARTIFACTS}/v3-r16"                       # config.json + checkpoints/best/
V4MIX_RUN_DIR = f"{H2_NZIW}/v4-mixed-r16"                   # config.json + checkpoints/best/
MANIFEST = f"{H2_NZIW}/NZiW4QH83CI/manifest.NZiW4QH83CI.jsonl"
CORRECTIONS = f"{H2_NZIW}/NZiW4QH83CI/corrections.NZiW4QH83CI.json"

# Packaging bug found 2026-08-17 running this exact notebook: the zip has
# audio at NZiW4QH83CI/audio/seg_*.wav (mirrors this repo's own
# dataset/youtube-meetings/audio/<meeting_id>/ layout), but manifest's
# audio_filepath is "NZiW4QH83CI/seg_0000.wav" -- one level shallower.
# Kaggle input is read-only, so flatten into /kaggle/working instead of
# re-uploading the 123 MB dataset.
import shutil
AUDIO_ROOT = "/kaggle/working/nziw_audio"
shutil.copytree(f"{H2_NZIW}/NZiW4QH83CI/audio", f"{AUDIO_ROOT}/NZiW4QH83CI", dirs_exist_ok=True)

print("V3_RUN_DIR/config.json exists:", os.path.exists(f"{V3_RUN_DIR}/config.json"))
print("V3_RUN_DIR/checkpoints/best exists:", os.path.exists(f"{V3_RUN_DIR}/checkpoints/best"))
print("V4MIX_RUN_DIR/config.json exists:", os.path.exists(f"{V4MIX_RUN_DIR}/config.json"))
print("V4MIX_RUN_DIR/checkpoints/best exists:", os.path.exists(f"{V4MIX_RUN_DIR}/checkpoints/best"))
print("MANIFEST exists:", os.path.exists(MANIFEST))
print("CORRECTIONS exists:", os.path.exists(CORRECTIONS))
print("one audio file exists:", os.path.exists(f"{AUDIO_ROOT}/NZiW4QH83CI/seg_0000.wav"))

## 4. Environment

No `HF_TOKEN` needed -- inference only, nothing is pushed.

In [ ]:
os.environ["TRANSFORMERS_AUTO_CONVERSION"] = "0"

## 5. Run the eval

Scores BOTH models on the same 20 corrected segments: v3-r16 (lambda=0.5)
and v4-mixed-r16 (lambda=0.25, v5's own published lambda) -- both from raw
(pre-lambda-bake) checkpoints via `src.lora.set_lambda`, same as
`stage_sweep_gate` uses for every lambda in a sweep.

**Untested on the author's machine** (no torch/GPU there).

In [ ]:
import subprocess

OUT_PREFIX = "/kaggle/working/predictions_NZiW4QH83CI"

cmd = [
    "python", "-m", "scripts.eval_v3_v5_on_corrected_segments",
    "--manifest", MANIFEST,
    "--corrections", CORRECTIONS,
    "--audio-root", AUDIO_ROOT,
    "--v3-run-dir", V3_RUN_DIR,
    "--v4mix-run-dir", V4MIX_RUN_DIR,
    "--out-prefix", OUT_PREFIX,
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise SystemExit(f"eval_v3_v5_on_corrected_segments failed with exit code {proc.returncode}")

## 6. Read the result

This is a THIRD data point alongside `first10.wav` (v5 worse, both CER and
retention) and tier4a (v5 better, both CER and retention) -- it does not by
itself resolve the contradiction, it adds one more real, held-out,
human-corrected reference on a third distinct domain (programming/RPA,
vs career-webinar and ML-lecture). Read all three together in the report,
don't pick whichever agrees with a preferred story.

`predictions_NZiW4QH83CI_v3.csv` / `_v4mix.csv` written to `/kaggle/working/` --
download before the session ends.